# Get Data

In [11]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import os
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Show all columns
pd.set_option('display.max_columns', 150)
pd.set_option('display.width', 100)

UTZ raises exception: Too Many Requests. Rate limited. Try after a while.VSTS raises exception: Too Many Requests. Rate limited. Try after a while.
EQBK raises exception: Too Many Requests. Rate limited. Try after a while.
PFLT raises exception: Too Many Requests. Rate limited. Try after a while.
BTQ raises exception: Too Many Requests. Rate limited. Try after a while.
HLX raises exception: Too Many Requests. Rate limited. Try after a while.
NTSK raises exception: Too Many Requests. Rate limited. Try after a while.
BCSF raises exception: Too Many Requests. Rate limited. Try after a while.
LOT raises exception: Too Many Requests. Rate limited. Try after a while.
FUBO raises exception: Too Many Requests. Rate limited. Try after a while.
NBN raises exception: Too Many Requests. Rate limited. Try after a while.
NKTR raises exception: Too Many Requests. Rate limited. Try after a while.
CPF raises exception: Too Many Requests. Rate limited. Try after a while.
MEG raises exception: Too Many R

In [12]:
# -----------------------------------------
# Get all US-traded stocks with market cap > $750M using FMP API
# Iterates by sector to avoid 1000 record limit per request
# -----------------------------------------

FMP_API_KEY = "UdRiHSXViYG3qI0ChXrlhVRvqOPXRAB5"

# All GICS sectors
SECTORS = [
    "Technology",
    "Healthcare", 
    "Financial Services",
    "Consumer Cyclical",
    "Communication Services",
    "Industrials",
    "Consumer Defensive",
    "Energy",
    "Basic Materials",
    "Real Estate",
    "Utilities"
]


def get_stocks_by_sector(sector, min_mcap_millions=750, api_key=FMP_API_KEY):
    """
    Get all US stocks in a specific sector above market cap threshold.
    Uses 1 API call per sector.
    """
    import time
    
    url = "https://financialmodelingprep.com/stable/company-screener"
    params = {
        "marketCapMoreThan": int(min_mcap_millions * 1e6),
        "isActivelyTrading": True,
        "isEtf": False,
        "isFund": False,
        "exchange": "NYSE,NASDAQ,AMEX",
        "sector": sector,
        "apikey": api_key
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        
        if response.status_code != 200:
            print(f"  Error for {sector}: {response.status_code}")
            return pd.DataFrame()
        
        data = response.json()
        
        if isinstance(data, dict) and 'Error Message' in data:
            print(f"  API Error for {sector}: {data['Error Message']}")
            return pd.DataFrame()
        
        df = pd.DataFrame(data)
        print(f"  {sector}: {len(df)} stocks")
        
        # Small delay to avoid rate limiting
        time.sleep(0.5)
        
        return df
        
    except Exception as e:
        print(f"  Exception for {sector}: {e}")
        return pd.DataFrame()


def get_us_stocks_fmp(min_mcap_millions=750, api_key=FMP_API_KEY):
    """
    Get all US stocks above market cap threshold by iterating through sectors.
    Uses ~11 API calls (1 per sector).
    """
    all_stocks = []
    
    print(f"Fetching stocks with market cap >= ${min_mcap_millions}M by sector...")
    print(f"This will use {len(SECTORS)} API calls.\n")
    
    for sector in SECTORS:
        df = get_stocks_by_sector(sector, min_mcap_millions, api_key)
        if not df.empty:
            all_stocks.append(df)
    
    if not all_stocks:
        raise Exception("No data retrieved from any sector")
    
    # Combine all sectors
    df = pd.concat(all_stocks, ignore_index=True)
    df.drop(columns=["exchange"])
    
    # Clean up columns
    df = df.rename(columns={
        'symbol': 'ticker',
        'companyName': 'name', 
        'marketCap': 'market_cap',
        'exchangeShortName': 'exchange'
    })
    
    # Select relevant columns (handle both old and new API response formats)
    possible_cols = ['ticker', 'name', 'market_cap', 'sector', 'industry', 'exchange', 
                     'country', 'price', 'beta', 'volume']
    cols = [c for c in possible_cols if c in df.columns]
    df = df[cols]
    
    # Remove duplicates (in case any stock appears in multiple sectors)
    df = df.drop_duplicates(subset=['ticker'])
    
    # Sort by market cap
    df = df.sort_values('market_cap', ascending=False).reset_index(drop=True)
    
    print(f"\n=== Total: {len(df)} unique stocks ===")
    return df


def load_or_fetch_universe(csv_path, min_mcap_millions=750, api_key=FMP_API_KEY, refresh=False):
    """
    Load ticker universe from CSV if it exists, otherwise fetch from FMP and save.
    """
    if os.path.exists(csv_path) and not refresh:
        print(f"Loading universe from {csv_path}")
        df = pd.read_csv(csv_path)
        print(f"Loaded {len(df)} tickers")
        return df
    
    print("Fetching fresh universe from FMP API...")
    df = get_us_stocks_fmp(min_mcap_millions, api_key)
    
    # Save to CSV
    df.to_csv(csv_path, index=False)
    print(f"Saved universe to {csv_path}")
    
    return df


# ============================================================
# Fetch and save the universe (uses ~11 API calls if CSV doesn't exist)
# ============================================================

UNIVERSE_CSV = "us_stocks_500m.csv"
MIN_MCAP = 500  # millions USD

# Set refresh=True to force a fresh fetch from FMP
stocks_df = load_or_fetch_universe(UNIVERSE_CSV, min_mcap_millions=MIN_MCAP, refresh=False)

display(stocks_df.head(10))
print(f"\nTotal tickers: {len(stocks_df)}")
print(f"\nSector distribution:")
display(stocks_df['sector'].value_counts())

Loading universe from us_stocks_500m.csv
Loaded 3098 tickers


,ticker,name,market_cap,sector,industry,exchange,exchange.1,country,price,beta,volume
0,NVDA,NVIDIA Corporation,4239786770335,Technology,Semiconductors,NASDAQ Global Select,NASDAQ,US,174.14,2.284,173661029
1,AAPL,Apple Inc.,4021975523070,Technology,Consumer Electronics,NASDAQ Global Select,NASDAQ,US,272.19,1.107,51509672
2,GOOG,Alphabet Inc.,3665542911390,Technology,Internet Content & Information,NASDAQ Global Select,NASDAQ,US,303.75,1.070,19859779
3,GOOGL,Alphabet Inc.,3649975667421,Technology,Internet Content & Information,NASDAQ Global Select,NASDAQ,US,302.46,1.070,32122646
4,MSFT,Microsoft Corporation,3597428078164,Technology,Software - Infrastructure,NASDAQ Global Select,NASDAQ,US,483.98,1.070,26736126
5,AMZN,"Amazon.com, Inc.",2423979947546,Consumer Cyclical,Specialty Retail,NASDAQ Global Select,NASDAQ,US,226.76,1.372,48340627
6,META,"Meta Platforms, Inc.",1674802780326,Technology,Internet Content & Information,NASDAQ Global Select,NASDAQ,US,664.45,1.273,19001162
7,TSLA,"Tesla, Inc.",1556592330390,Consumer Cyclical,Auto - Manufacturers,NASDAQ Global Select,NASDAQ,US,483.37,1.878,94595726
8,AVGO,Broadcom Inc.,1554436009092,Technology,Semiconductors,NASDAQ Global Select,NASDAQ,US,329.88,1.204,51123858
9,TSM,Taiwan Semiconductor Manufacturing Company Lim...,1476499408065,Technology,Semiconductors,New York Stock Exchange,NYSE,TW,284.68,1.267,11182194



Total tickers: 3098

Sector distribution:


sector
Financial Services        557
Technology                446
Healthcare                445
Industrials               400
Consumer Cyclical         319
Real Estate               188
Energy                    187
Basic Materials           178
Communication Services    139
Consumer Defensive        132
Utilities                 107
Name: count, dtype: int64

In [13]:
# -----------------------------------------
# Fetch OHLCV data using yfinance (free, unlimited)
# -----------------------------------------
import time

def get_stock_ohlcv(ticker, start_date, end_date, interval="1d"):
    """
    Downloads OHLCV data for a single ticker from Yahoo Finance.
    
    Args:
        ticker (str): Stock ticker symbol
        start_date (str): "YYYY-MM-DD"
        end_date (str): "YYYY-MM-DD"
        interval (str): Data frequency
    
    Returns:
        pandas.DataFrame: OHLCV data
    """
    try:
        stock = yf.Ticker(ticker)
        df = stock.history(start=start_date, end=end_date, interval=interval)
        if df.empty:
            print(f'{ticker} empty')
        if not df.empty:
            df = df.drop(columns=['Dividends', 'Stock Splits'], errors='ignore')
        return df
    except Exception as e:
        print(f'{ticker} raises exception: {e}')
        return pd.DataFrame()


def get_all_stocks_ohlcv(tickers, start_date, end_date, interval="1d", max_workers=5, sleep_time=0.2):
    """
    Downloads OHLCV data for multiple tickers in parallel using yfinance.
    Includes sleep between requests to avoid rate limiting.
    
    Args:
        tickers (list): List of ticker symbols
        start_date (str): "YYYY-MM-DD"
        end_date (str): "YYYY-MM-DD"
        interval (str): Data frequency
        max_workers (int): Number of parallel threads (lower = less rate limiting)
        sleep_time (float): Seconds to sleep between each request
    
    Returns:
        dict: Dictionary mapping ticker -> DataFrame
    """
    all_data = {}
    failed_tickers = []
    
    def fetch_ticker(ticker):
        time.sleep(sleep_time)  # Sleep to avoid rate limiting
        df = get_stock_ohlcv(ticker, start_date, end_date, interval)
        return ticker, df
    
    print(f"Fetching OHLCV data for {len(tickers)} tickers from {start_date} to {end_date}...")
    print(f"Using {max_workers} workers with {sleep_time}s delay between requests...")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_ticker, ticker): ticker for ticker in tickers}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading"):
            ticker, df = future.result()
            if not df.empty:
                all_data[ticker] = df
            else:
                failed_tickers.append(ticker)
    
    print(f"Successfully fetched: {len(all_data)} tickers")
    print(f"Failed/empty: {len(failed_tickers)} tickers")
    
    return all_data, failed_tickers


def calculate_returns(stock_data, periods=[1, 5, 10, 21]):
    """
    Calculate log returns for all tickers at specified periods.
    
    Args:
        stock_data (dict): Dictionary mapping ticker -> OHLCV DataFrame
        periods (list): List of periods for return calculation (in trading days)
    
    Returns:
        pd.DataFrame: Wide DataFrame with returns for all tickers
    """
    returns_list = []
    
    for ticker, df in stock_data.items():
        if df.empty or 'Close' not in df.columns:
            continue
            
        ticker_returns = pd.DataFrame(index=df.index)
        ticker_returns['ticker'] = ticker
        
        for period in periods:
            ticker_returns[f'ret_{period}d'] = np.log(df['Close'] / df['Close'].shift(period))
        
        returns_list.append(ticker_returns)
    
    if not returns_list:
        return pd.DataFrame()
    
    # Combine all returns
    all_returns = pd.concat(returns_list, axis=0)
    all_returns = all_returns.reset_index().rename(columns={'index': 'date', 'Date': 'date'})
    
    return all_returns


def save_to_csv(df, csv_path):
    """Save returns DataFrame to CSV."""
    df.to_csv(csv_path, index=False)
    print(f"Saved returns data to {csv_path}")


# ============================================================
# Fetch OHLCV data and calculate returns
# ============================================================

# Configuration
START_DATE = "2018-01-01"
END_DATE = "2026-02-13"
RETURN_PERIODS = [1, 5, 10, 21, 63]  # 1d, 1wk, 2wk, 1mo, 3mo
OHLCV_CSV = "stock_ohlcv_data.csv"
RETURNS_CSV = "stock_returns.csv"

# Get list of tickers from universe
tickers = stocks_df['ticker'].tolist()

# Fetch all OHLCV data with rate limiting protection
stock_data, failed_tickers = get_all_stocks_ohlcv(
    tickers, 
    START_DATE, 
    END_DATE, 
    interval="1d", 
    max_workers=1,
    sleep_time=0.5
)

# Calculate returns
returns_df = calculate_returns(stock_data, periods=RETURN_PERIODS)
print(f"\nReturns DataFrame shape: {returns_df.shape}")

# Save returns to CSV
save_to_csv(returns_df, RETURNS_CSV)

# Show summary
print(f"\n=== Summary ===")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Tickers with data: {returns_df['ticker'].nunique()}")
print(f"Total rows: {len(returns_df)}")
print(f"Return periods: {RETURN_PERIODS}")

Fetching OHLCV data for 3098 tickers from 2018-01-01 to 2026-02-13...
Using 1 workers with 0.5s delay between requests...


Downloading:   5%|▍         | 144/3098 [01:33<31:25,  1.57it/s]

CMCSV empty


Downloading:   5%|▌         | 161/3098 [01:44<32:08,  1.52it/s]

MMC empty


...

NVAWW empty


Downloading:  99%|█████████▉| 3073/3098 [32:50<00:15,  1.58it/s]

CLCO empty


Downloading: 100%|██████████| 3098/3098 [33:07<00:00,  1.56it/s]


Successfully fetched: 3056 tickers
Failed/empty: 42 tickers

Returns DataFrame shape: (5387851, 7)
Saved returns data to stock_returns.csv

=== Summary ===
Date range: 2018-01-01 to 2026-02-13
Tickers with data: 3056
Total rows: 5387851
Return periods: [1, 5, 10, 21, 63]


In [14]:
# def stock_data_to_df(stock_data):
#     """
#     Convert stock_data dict to a single DataFrame with ticker column.
#     """
#     dfs = []
#     for ticker, df in stock_data.items():
#         if df.empty:
#             continue
#         df = df.copy()
#         df['ticker'] = ticker
#         df = df.reset_index().rename(columns={'index': 'date', 'Date': 'date'})
#         dfs.append(df)
    
#     if not dfs:
#         return pd.DataFrame()
    
#     combined = pd.concat(dfs, ignore_index=True)
#     cols = ['date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
#     cols = [c for c in cols if c in combined.columns]
#     return combined[cols]


# if len(failed_tickers) > 0:
#     print(f"Retrying {len(failed_tickers)} failed tickers...")
#     print("Waiting 30 seconds before retry to avoid rate limiting...")
#     time.sleep(30)
    
#     # Retry failed tickers
#     retry_data, still_failed = get_all_stocks_ohlcv(
#         failed_tickers, 
#         START_DATE, 
#         END_DATE, 
#         interval="1d", 
#         max_workers=20
#     )
    
#     # If we got new data, append to CSV
#     if len(retry_data) > 0:
#         retry_df = stock_data_to_df(retry_data)
#         print(f"\nRetry recovered: {len(retry_data)} tickers")
        
#         # Append to existing CSV
#         retry_df.to_csv(OHLCV_CSV, mode='a', header=False, index=False)
#         print(f"Appended retry data to {OHLCV_CSV}")
        
#         # Update stock_data with recovered tickers
#         stock_data.update(retry_data)
        
#         # Update failed_tickers list
#         failed_tickers = still_failed
    
#     print(f"\n=== After Retry ===")
#     print(f"Total successful: {len(stock_data)} tickers")
#     print(f"Still failed: {len(failed_tickers)} tickers")
# else:
#     print("No failed tickers to retry!")


In [15]:
# -----------------------------------------
# Convert stock_data dict to DataFrame and save to CSV
# -----------------------------------------

def stock_data_to_df(stock_data):
    """
    Convert stock_data dict to a single DataFrame with ticker column.
    
    Args:
        stock_data (dict): Dictionary mapping ticker -> OHLCV DataFrame
    
    Returns:
        pd.DataFrame: Combined DataFrame with columns: date, ticker, Open, High, Low, Close, Volume
    """
    dfs = []
    for ticker, df in stock_data.items():
        if df.empty:
            continue
        df = df.copy()
        df['ticker'] = ticker
        df = df.reset_index().rename(columns={'index': 'date', 'Date': 'date'})
        dfs.append(df)
    
    if not dfs:
        return pd.DataFrame()
    
    combined = pd.concat(dfs, ignore_index=True)
    # Reorder columns
    cols = ['date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
    cols = [c for c in cols if c in combined.columns]
    return combined[cols]


# Convert and save OHLCV data
ohlcv_df = stock_data_to_df(stock_data)
print(f"OHLCV DataFrame shape: {ohlcv_df.shape}")
print(f"Unique tickers: {ohlcv_df['ticker'].nunique()}")

# Save to CSV
ohlcv_df.to_csv(OHLCV_CSV, index=False)
print(f"\nSaved OHLCV data to {OHLCV_CSV}")

OHLCV DataFrame shape: (5387851, 7)
Unique tickers: 3056

Saved OHLCV data to stock_ohlcv_data.csv


In [16]:
# -----------------------------------------
# Analyze failed tickers by market cap
# -----------------------------------------

# Get failed tickers info from universe
failed_df = stocks_df[stocks_df['ticker'].isin(failed_tickers)].copy()

print(f"Total failed tickers: {len(failed_tickers)}")
print(f"Failed tickers found in universe: {len(failed_df)}")

# Analyze by market cap thresholds
mcap_thresholds = [500, 750, 1000, 2000, 5000, 10000]  # in millions

print("\n=== Failed Tickers by Market Cap ===")
for threshold in mcap_thresholds:
    threshold_dollars = threshold * 1e6
    count = len(failed_df[failed_df['market_cap'] >= threshold_dollars])
    print(f"Market cap >= ${threshold}M: {count} failed tickers")

# Show top failed tickers by market cap
print("\n=== Top 20 Failed Tickers by Market Cap ===")
display(failed_df.nlargest(20, 'market_cap')[['ticker', 'name', 'market_cap', 'sector', 'industry']])

# Sector breakdown of failed tickers
print("\n=== Failed Tickers by Sector ===")
display(failed_df['sector'].value_counts())

Total failed tickers: 42
Failed tickers found in universe: 42

=== Failed Tickers by Market Cap ===
Market cap >= $500M: 42 failed tickers
Market cap >= $750M: 35 failed tickers
Market cap >= $1000M: 28 failed tickers
Market cap >= $2000M: 22 failed tickers
Market cap >= $5000M: 12 failed tickers
Market cap >= $10000M: 4 failed tickers

=== Top 20 Failed Tickers by Market Cap ===


,ticker,name,market_cap,sector,industry
143,CMCSV,Comcast Corporation Class A Common Stock Ex-Di...,103527671695,Consumer Cyclical,Broadcasting
160,MMC,"Marsh & McLennan Companies, Inc.",91379602780,Financial Services,Insurance - Brokers
505,EBR,Centrais Elétricas Brasileiras S.A. - Eletrobrás,25577232356,Utilities,Regulated Electric
855,ERJ,Embraer S.A.,11845015303,Industrials,Aerospace & Defense
964,RNA,"Avidity Biosciences, Inc.",9467378489,Healthcare,Biotechnology
1115,ELP,Companhia Paranaense de Energia - COPEL,7394384487,Utilities,Diversified Utilities
1122,MRUS,Merus N.V.,7341469300,Healthcare,Biotechnology
1126,SNV,Synovus Financial Corp.,7253110241,Financial Services,Banks - Regional
1163,ZK,ZEEKR Intelligent Technology Holding Limited,6814196533,Consumer Cyclical,Auto - Manufacturers
1165,VSNTV,"Versant Media Group, Inc. Class A Common Stock...",6775050893,Communication Services,Entertainment



=== Failed Tickers by Sector ===


sector
Consumer Cyclical         7
Financial Services        7
Industrials               6
Healthcare                6
Communication Services    4
Technology                4
Utilities                 3
Real Estate               3
Energy                    1
Basic Materials           1
Name: count, dtype: int64